# Pre-train AMRBART with MBart-50 (Vietnamese)

This notebook runs the 6-task AMR pre-training on Google Colab with a single GPU.

## 1. Mount Google Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Clone repository

In [4]:
!rm -rf /content/AMRBART
!git clone -b feat/mbart-50 https://github.com/Phucgiacat/AMRBART.git /content/AMRBART
!ls /content/AMRBART/pre-train/

Cloning into '/content/AMRBART'...
remote: Enumerating objects: 867, done.
remote: Counting objects: 100% (551/551), done.
remote: Compressing objects: 100% (293/293), done.
remote: Total 867 (delta 319), reused 485 (delta 255), pack-reused 316 (from 1)
Receiving objects: 100% (867/867), 7.20 MiB | 16.57 MiB/s, done.
Resolving deltas: 100% (502/502), done.
common
data_interface
model_interface
Pretrain_Mbart50.ipynb
requirements.txt
run_multitask_unified_pretraining.py
run-posttrain-bart-textinf-joint-denoising-6task-large-unified-A100.sh
run-posttrain-bart-textinf-joint-denoising-6task-large-unified-V100.sh
run-posttrain-mbart50-vietnamese-6task-large-unified.sh


## 3. Install dependencies

In [ ]:
# Add deadsnakes PPA and install Python 3.8
!apt-get update -qq 2>/dev/null
!apt-get install -qq -y software-properties-common 2>/dev/null
!add-apt-repository -y ppa:deadsnakes/ppa 2>/dev/null
!apt-get update -qq 2>/dev/null
!apt-get install -qq -y python3.8 python3.8-venv python3.8-dev python3.8-distutils

# Create a Python 3.8 venv
!python3.8 -m venv /content/py38env
!/content/py38env/bin/python --version

# Upgrade pip
!/content/py38env/bin/pip install --upgrade pip

# Install PyTorch 1.8.1 + CUDA 11.1 (Colab GPU driver is backward-compatible)
!/content/py38env/bin/pip install torch==1.8.1+cu111 torchvision==0.9.1+cu111 torchaudio==0.8.1 \
  -f https://download.pytorch.org/whl/torch_stable.html

# Install all other dependencies
!/content/py38env/bin/pip install -r /content/AMRBART/pre-train/requirements.txt

Repository: 'deb https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu/ jammy main'
Description:
This PPA contains more recent Python versions packaged for Ubuntu.

Disclaimer: there's no guarantee of timely updates in case of security problems or other issues. If you want to use them in a security-or-otherwise-critical environment (say, on a production server), you do so at your own risk.

Update Note
Please use this repository instead of ppa:fkrull/deadsnakes.

Reporting Issues

Issues can be reported in the master issue tracker at:
https://github.com/deadsnakes/issues/issues

Supported Ubuntu and Python Versions

- Ubuntu 22.04 (jammy) Python3.7 - Python3.9, Python3.11 - Python3.13
- Ubuntu 24.04 (noble) Python3.7 - Python3.11, Python3.13
- Note: Python 3.10 (jammy), Python3.12 (noble) are not provided by deadsnakes as upstream ubuntu provides those packages.

Why some packages aren't built:
- Note: for jammy and noble, older python versions requre libssl<3 so they are not currentl

In [5]:
%cd /content/AMRBART/pre-train/

/content/AMRBART/pre-train


In [6]:
!pip install uv
!sed -i '/python/d' requirements.txt

!rm -rf .venv
!uv venv .venv --python 3.8

!uv pip install --python .venv -r requirements.txt
!uv pip install --python .venv penman evaluate smatch rouge_score sacrebleu tqdm
!pip install --upgrade transformers huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 85.6 MB/s eta 0:00:00
Using CPython 3.8.20
Creating virtual environment at: .venv
Activate with: source .venv/bin/activate
Resolved 135 packages in 5.23s
Prepared 131 packages in 2m 29s
Installed 135 packages in 865ms
 + absl-py==2.3.1
 + aiohappyeyeballs==2.4.4
 + aiohttp==3.10.11
 + aiosignal==1.3.1
 + amrlib==0.8.1
 + annotated-types==0.7.0
 + async-timeout==5.0.1
 + attrs==25.3.0
 + blis==0.7.11
 + cached-property==2.0.1
 + catalogue==2.0.10
 + certifi==2026.2.25
 + cffi==1.17.1
 + charset-normalizer==3.4.7
 + click==8.1.8
 + cloudpathlib==0.20.0
 + colorama==0.4.6
 + confection==0.1.5
 + configargparse==1.7.5
 + cryptography==46.0.7
 + cymem==2.0.11
 + datasets==2.4.0
 + dill==0.3.5.1
 + eval-type-backport==0.3.1
 + filelock==3.16.1
 + frozenlist==1.5.0
 + fsspec==2025.3.0
 + gensim==4.3.3
 + gitdb==4.0.12
 + gitpython==3.1.46
 + google-auth==2.49.1
 + google-auth-oauthlib==1.0.0
 + grpcio==1.70.0
 + h5py-cache==1.0
 + hf-xe

## 4. Download & prepare data

In [7]:
!pip install -q gdown
!gdown --folder https://drive.google.com/drive/folders/10gwxpxAfha9zd1q6nakBBMohSBXGmmbC?usp=sharing -O /content/data

Retrieving folder contents
Processing file 1EOgd_xwTSsPFxMiE5ghCcNTx6y1dpDzz dev.amr
Processing file 1oHWsyhzKgFGhwm14LfzBTiE5Lo6c-wCn dev.jsonl
Processing file 1nyoJ_eYI9ikfkxvPK3MLWzAEmBk8qF97 dev.txt
Processing file 1fsVyEX7Fyv3DS8y92d6peAAOyzWEkabv test.amr
Processing file 10u4EXv9McDUFzr8rbviWHU7e3cU7fyNb test.jsonl
Processing file 1YsOt4CZCs6SVef23wFaAFZ4bXwTE3c1R test.txt
Processing file 1Jk95japz4vKmdEm6KYxFcKQX1U1NQDnS train.amr
Processing file 18BIpZ80R0tJAEnhnLlwu4XfC1UwwiWRV train.jsonl
Processing file 1n92pUN1QFAV6-FI4Eim54xJhDSrwWRqR train.txt
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1EOgd_xwTSsPFxMiE5ghCcNTx6y1dpDzz
To: /content/data/dev.amr
100% 395k/395k [00:00<00:00, 5.85MB/s]
Downloading...
From: https://drive.google.com/uc?id=1oHWsyhzKgFGhwm14LfzBTiE5Lo6c-wCn
To: /content/data/dev.jsonl
100% 674k/674k [00:00<00:00, 7.54MB/s]
Downloading...
From: https:

In [8]:
!mkdir -p /content/AMRBART/pre-train/data/ViAMR
!cp /content/data/*.jsonl /content/AMRBART/pre-train/data/ViAMR/
!cp /content/AMRBART/pre-train/data/ViAMR/dev.jsonl /content/AMRBART/pre-train/data/ViAMR/val.jsonl
!ls -la /content/AMRBART/pre-train/data/ViAMR/

total 7368
drwxr-xr-x 2 root root    4096 Apr  8 17:04 .
drwxr-xr-x 3 root root    4096 Apr  8 17:04 ..
-rw-r--r-- 1 root root  673559 Apr  8 17:04 dev.jsonl
-rw-r--r-- 1 root root  684068 Apr  8 17:04 test.jsonl
-rw-r--r-- 1 root root 5493661 Apr  8 17:04 train.jsonl
-rw-r--r-- 1 root root  673559 Apr  8 17:04 val.jsonl


## 5. Download MBart-50 model

In [9]:
!pip install -q huggingface_hub
!hf download facebook/mbart-large-50 --local-dir /content/mbart-large-50

Fetching 9 files:   0% 0/9 [00:00<?, ?it/s]Still waiting to acquire lock on /content/mbart-large-50/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)
Still waiting to acquire lock on /content/mbart-large-50/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)
Still waiting to acquire lock on /content/mbart-large-50/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)
Still waiting to acquire lock on /content/mbart-large-50/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)
Still waiting to acquire lock on /content/mbart-large-50/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)
Fetching 9 files: 100% 9/9 [00:43<00:00,  4.87s/it]
Download complete: : 4.89GB [00:43, 151MB/s]                              /content/mbart-large-50
Download complete: : 4.89GB [00:43, 112MB/s]


## 6. Run pre-training

In [ ]:
# Install sentencepiece INSIDE the .venv (not system Python)
!.venv/bin/pip install sentencepiece

In [ ]:
!cd /content/AMRBART/pre-train && bash run-posttrain-mbart50-vietnamese-6task-large-unified.sh